# Recomendación de contenido e híbrida V2

## Objetivo y alcance

Este cuaderno documenta el recomendador de contenido y la fusión híbrida sobre el catálogo Goodreads V2. No reentrena la factorización matricial ni reconstruye TF-IDF: carga artefactos versionados para inspección, evaluación e inferencia reproducibles. Goodreads aporta metadatos y preferencias históricas; no representa disponibilidad de ejemplares de la biblioteca.

## Datos, unidades y prevención de fuga

La unidad de catálogo es `book_id`; el índice colaborativo usa 566,739 libros con embedding y el índice de contenido incluye 931,048 libros. Los perfiles de contenido se construyen solo con ratings explícitos de al menos cuatro estrellas de `train`. Popularidad, perfiles y pesos se calculan sin usar `test`. La prueba histórica permanece como referencia y no interviene en la selección actual.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

root = Path.cwd()
if not (root / 'artifacts').exists():
    root = root.parent
sys.path.insert(0, str(root / 'src'))

content_summary = json.loads((root / 'artifacts/v2/content_tfidf/summary.json').read_text())
hybrid_config = json.loads((root / 'artifacts/v2/hybrid/config.json').read_text())
validation = pd.read_csv(root / 'artifacts/v2/hybrid/validation_catalog_universes.csv')
calibration = pd.read_csv(root / 'artifacts/v2/hybrid/calibration_full_catalog_validation.csv')
benchmark = json.loads((root / 'artifacts/v2/hybrid/inference_benchmark.json').read_text())
pd.DataFrame([content_summary]), pd.DataFrame([hybrid_config])

## Modelo de contenido: TF-IDF y coseno

Cada documento concatena título (con doble peso), descripción, nombres de autor, géneros temáticos e idioma. Se excluyen etiquetas de estantería que representan estado o propiedad, como `to-read`, `read` y `owned`. `TfidfVectorizer` genera una matriz CSR `float32` con unigramas y bigramas, 80,000 términos y norma L2. Por ello, el producto punto entre un perfil normalizado y un libro equivale a similitud coseno. No se materializa una matriz densa libro-libro.

In [ ]:
pd.Series({
    'libros indexados': content_summary['books'],
    'libros sin descripción': content_summary['books_without_description'],
    'términos': content_summary['matrix_shape'][1],
    'pesos TF-IDF no nulos': content_summary['matrix_nnz'],
    'memoria CSR (GiB)': content_summary['matrix_bytes'] / 2**30,
})

## Perfil de usuario y fusión

Para un usuario con preferencias positivas, el perfil es el promedio de las filas TF-IDF de sus libros positivos y se normaliza L2. En el universo colaborativo, cada componente se normaliza min-max sobre exactamente los libros candidatos no vistos de ese usuario: `h = w * norm(contenido) + (1 - w) * norm(colaborativo)`. Esta es la misma regla de la evaluación completa y de `RecommendationEngine.recommend_hybrid`.

Los 364,309 libros sin embedding no reciben un score colaborativo imputado. Se exponen en `recommend_content_only`, con `score_scope='content_only_catalog'`. Por tanto, sus scores no se comparan ni se intercalan con el ranking híbrido `score_scope='collaborative_catalog'`.

## Calibración solo con validación

La calibración actual recorre 100 usuarios de validación contra los 566,739 libros colaborativos elegibles. Este es el único universo donde contenido y colaborativo están disponibles para la misma lista de candidatos. El peso se selecciona por NDCG@10, con desempate por Recall@10 y menor peso de contenido.

In [ ]:
hybrid_curve = calibration.query("model == 'hybrid'").sort_values('content_weight')
ax = hybrid_curve.plot(x='content_weight', y='ndcg_at_k', marker='o', legend=False, figsize=(7, 4))
ax.set(xlabel='Peso de contenido', ylabel='NDCG@10 en validación', title='Calibración híbrida: catálogo colaborativo completo')
plt.show()
hybrid_curve[['content_weight', 'recall_at_k', 'ndcg_at_k']]

## Cobertura y evaluación por universo

La evaluación separa el catálogo colaborativo (566,739 libros; 60.87% de cobertura respecto al índice de contenido) del catálogo completo de contenido (931,048 libros; 100%). Popularidad, colaborativo, contenido e híbrido se comparan solo dentro del primer universo. En el segundo se informa contenido, pues colaborar o fusionar exigiría fabricar una señal inexistente. Estas métricas son una muestra de 100 usuarios y no permiten conclusiones causales ni sustituyen evaluación de cold start.

In [ ]:
validation[['model', 'content_weight', 'recall_at_k', 'ndcg_at_k', 'candidate_universe', 'catalog_coverage']]
plot_data = validation.assign(label=validation['model'] + ' | ' + validation['candidate_universe'].str.replace('_', ' '))
ax = plot_data.plot.barh(x='label', y='ndcg_at_k', legend=False, figsize=(9, 4))
ax.set(xlabel='NDCG@10', ylabel='', title='Validación por universo de candidatos')
plt.show()

## Ejemplos de inferencia

El motor reutiliza mappings y metadatos cargados durante inicialización. Para un usuario sin ratings positivos, el híbrido degrada a ranking colaborativo y `recommend_content_only` informa que no puede construir perfil. Los resultados incluyen título, autores y géneros para trazabilidad.

In [ ]:
from ml.inference import RecommendationEngine

engine = RecommendationEngine(root, version='v2')
example_user = next(user_id for user_id, user_idx in engine.user_to_idx.items() if engine._profile_for_user(int(user_idx)) is not None)
hybrid_example = engine.recommend_hybrid(example_user, k=5)
content_only_example = engine.recommend_content_only(example_user, k=5)
hybrid_example, content_only_example

## Rendimiento y limitaciones

El benchmark usa los artefactos V2 reales y registra tiempo de inicialización, RSS y latencia. La matriz TF-IDF permanece residente, por lo que el coste de inicialización y memoria debe considerarse al integrar un servicio. El motor no filtra disponibilidad física, reservas, edad recomendada ni licencias; esos criterios corresponden a la capa de biblioteca. Los resultados históricos de test no se usan para recalibrar el peso actual.

In [ ]:
pd.DataFrame(benchmark['operations']).assign(rss_after_gib=lambda frame: frame.rss_after_bytes / 2**30)
pd.DataFrame([benchmark['initialization']]).assign(rss_after_gib=lambda frame: frame.rss_after_bytes / 2**30)